[pretrain](https://github.com/OpenGVLab/VideoMAEv2/blob/master/models/modeling_pretrain.py)

In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid
from torchcodec.encoders import VideoEncoder

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../../')

In [2]:
from pathlib import Path

import torch
import numpy as np
from PIL import Image

%load_ext autoreload
%autoreload 2
    
from computer_vision.video_mae.parameter_parser import parser
from computer_vision.video_mae.dataset.pretrained_datasets import HybridVideoMAE, DataAugmentationForVideoMAEv2
# from computer_vision.video_mae.models.modeling_pretrain import pretrain_videomae_large_patch16_224
from computer_vision.video_mae.models.modeling_pretrain import PretrainVisionTransformerEncoder, PretrainVisionTransformerDecoder, \
PretrainVisionTransformer

In [3]:
from functools import partial

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as cp



In [4]:
encoder=PretrainVisionTransformerEncoder(img_size=224, patch_size=16, num_classes=1000, embed_dim=384, depth=12, num_heads=6, mlp_ratio=4., 
                                         qkv_bias=True, drop_path_rate=0., norm_layer=partial(nn.LayerNorm,eps=1e-6), init_values=0., 
                                         tubelet_size=2, use_learnable_pos_emb=False, with_cp=False, all_frames=16, cos_attn=False)
x=torch.rand(4,3,16,224,224)
mask=torch.rand(1, 1568)>0.5
mask=mask.expand(4,-1)
print(f'{mask.dtype=}, {mask.shape=}')
out=encoder(x, mask)
print(f"{out.shape=}")
nn.MSELoss()(out, torch.rand_like(out)).backward()

all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
mask.dtype=torch.bool, mask.shape=torch.Size([4, 1568])
out.shape=torch.Size([4, 763, 1000])


In [5]:
# tiny
embed_dim=192 # or 128 (ultra-light)
depth=12 # or 8 (ultra-light)
num_heads=3 # keep dim_per_head at 64
mlp_ratio=3. # 2 (ultra-light)
# increase patch_size to 32 (ultra-light), reducing sequence length by 4x
# increase tubelet to 4, reduce sequence length by 2x
# Note: transformer complexity is quadratic relative to the sequence length

# for performance, use cos_attn=True and init_values=1e-5 (former stabalize training, latter helps deeper, narrower models converge faster)
encoder=PretrainVisionTransformerEncoder(img_size=224, patch_size=16, num_classes=1000, embed_dim=embed_dim, depth=depth, num_heads=num_heads, 
                                         mlp_ratio=mlp_ratio, qkv_bias=True, drop_path_rate=0., norm_layer=partial(nn.LayerNorm,eps=1e-6), init_values=0., 
                                         tubelet_size=2, use_learnable_pos_emb=False, with_cp=False, all_frames=16, cos_attn=False)
x=torch.rand(4,3,16,224,224)
mask=torch.rand(1, 1568)>0.5
mask=mask.expand(4,-1)
print(f'{mask.dtype=}, {mask.shape=}')
out=encoder(x, mask)
print(f"{out.shape=}")
nn.MSELoss()(out, torch.rand_like(out)).backward()

all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
mask.dtype=torch.bool, mask.shape=torch.Size([4, 1568])
out.shape=torch.Size([4, 780, 1000])


In [7]:
# tiny settings
encoder_embed_dim=192 # 128 ultra-light  --> 192/3=64 dim per head
encoder_depth=12 # or 8 for speed or ultra-light
encoder_num_heads=3 # 2 ultra-light
decoder_embed_dim=128 # 96 ultra-light
decoder_depth=4 # 2 ultra-light
decoder_num_heads=2
mlp_ratio=3. # 2. ultra-light
init_values=1e-5 # if the model loss is oscilatting or not decreasing, increase this to 1e-4 to allow more signal through the residual branches

patch_size=16
tubelet_size=2
decoder_num_classes=3*tubelet_size*(patch_size**2)
print(f"{decoder_num_classes=}")
model=PretrainVisionTransformer(img_size=224, patch_size=patch_size, encoder_in_chans=3, encoder_num_classes=0, encoder_embed_dim=encoder_embed_dim, 
                                encoder_depth=encoder_depth, encoder_num_heads=encoder_num_heads, decoder_num_classes=decoder_num_classes, 
                                decoder_embed_dim=decoder_embed_dim, decoder_depth=decoder_depth, decoder_num_heads=decoder_num_heads, 
                                mlp_ratio=mlp_ratio, qkv_bias=True, qk_scale=None, drop_rate=0., attn_drop_rate=0., drop_path_rate=0., 
                                norm_layer=nn.LayerNorm, init_values=1e-5, use_learnable_pos_emb=False, tubelet_size=tubelet_size,
                                with_cp=False, all_frames=16, cos_attn=False, attn_head_dim=None)

x=torch.rand(4,3,16,224,224)
mask=torch.rand(1, 1568)>0.10 # drop about 90%
mask=mask.expand(4,-1)
decode_mask=None
print(f'{mask.dtype=}, {mask.shape=}, {mask.sum()/mask.numel() =}')
out=model(x, mask, decode_mask=decode_mask)
print(f"{out.shape=}")
nn.MSELoss()(out, torch.rand_like(out)).backward()

decoder_num_classes=1536
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
all_head_dim//self.num_heads=64
mask.dtype=torch.bool, mask.shape=torch.Size([4, 1568]), mask.sum()/mask.numel() =tensor(0.8941)
out.shape=torch.Size([4, 1402, 1536])
